In [3]:

print("Pyproj version:", pyproj.__version__)
print("Data directory:", pyproj.datadir.get_data_dir())

Pyproj version: 3.7.2
Data directory: C:\Users\Administrator\.local\share\mamba\envs\mgr\Library\share\proj


In [2]:
import geopandas as gpd
from sqlalchemy import create_engine, text
import pyproj

In [14]:
TARGET_CRS = 3035
layers = {
    "niecki_filled": ("../data/detected_subsidence_areas/detected_areas_xh_8_fill.shp"),
    "DSA": ("../data/detected_subsidence_areas/detected_areas_xh_8.shp")
}


In [15]:
db_user = "postgres"
db_password = "postgres"
db_host = "localhost"
db_port = "5432"
db_name = "niecki"

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

In [18]:
def load_layer(table, path):
    gdf = gpd.read_file(path)
    gdf = gdf.set_crs(TARGET_CRS)
    gdf.columns = [c.lower() for c in gdf.columns]   # COType -> cotype
    gdf["geometry"] = gdf.geometry.make_valid()
    gdf = gdf.rename_geometry("geom")
    gdf.to_postgis(table, engine, if_exists="replace", index=False)
    with engine.begin() as con:
        con.execute(text(f"CREATE INDEX IF NOT EXISTS {table}_geom_gix ON {table} USING GIST (geom)"))
        con.execute(text(f"ANALYZE {table}"))

In [19]:
for t, p in layers.items():
    load_layer(t, p)

ProgrammingError: (psycopg2.errors.UndefinedTable) BŁĄD:  relacja "dsa" nie istnieje

[SQL: CREATE INDEX IF NOT EXISTS DSA_geom_gix ON DSA USING GIST (geom)]
(Background on this error at: https://sqlalche.me/e/20/f405)